In [2]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data', exist_ok=True)

np.random.seed(42)

N = 60000

print("Generating Student Loan dataset...")

# Student profile
age = np.random.randint(21, 32, N)

gender = np.random.choice(
    ['Male', 'Female'],
    N,
    p=[0.58, 0.42]
)

course_type = np.random.choice(
    ['Engineering', 'MBA', 'Medical', 'Law', 'Others'],
    N,
    p=[0.38, 0.22, 0.15, 0.10, 0.15]
)

institute_tier = np.random.choice(
    ['Tier1', 'Tier2', 'Tier3'],
    N,
    p=[0.20, 0.45, 0.35]
)

gpa = np.random.uniform(5.5, 10.0, N).round(2)

gpa_trend = np.random.choice(
    ['Improving', 'Stable', 'Declining'],
    N,
    p=[0.25, 0.50, 0.25]
)

# Loan features
loan_amt = np.random.lognormal(
    12.8,
    0.6,
    N
).round(-3).clip(100000, 5000000)

interest_rate = np.random.uniform(
    9.5,
    14.5,
    N
).round(2)

tenure_mo = np.random.choice(
    [60, 84, 120, 144],
    N,
    p=[0.15, 0.30, 0.35, 0.20]
)

moratorium_mo = np.random.choice(
    [6, 12, 18, 24],
    N,
    p=[0.20, 0.40, 0.25, 0.15]
)

emi = (
    loan_amt * (interest_rate / 1200) *
    (1 + interest_rate / 1200) ** tenure_mo /
    (
        (1 + interest_rate / 1200) ** tenure_mo - 1
    )
).round(0)

# Post-graduation employment
employed = np.random.binomial(1, 0.72, N)

salary_mo = np.where(
    employed == 1,
    np.random.lognormal(10.5, 0.6, N).clip(15000, 500000),
    np.random.lognormal(9.2, 0.5, N).clip(5000, 50000)
).round(0)

emi_income = (emi / salary_mo).round(3)

# Repayment behavior
months_since_disbursement = np.random.randint(1, 60, N)

missed_pmts_past = np.random.poisson(0.4, N).clip(0, 8)

payment_delay_avg = np.random.exponential(
    5,
    N
).clip(0, 30).round(1)

auto_debit = np.random.binomial(1, 0.55, N)

co_borrower = np.random.binomial(1, 0.68, N)

family_income_mo = np.random.lognormal(
    11.0,
    0.8,
    N
).round(0)

# Default probabilities (3 horizons)
base_risk = (
    0.06
    + 0.12 * (gpa < 6.5).astype(float)
    + 0.10 * (gpa_trend == 'Declining').astype(float)
    - 0.08 * (institute_tier == 'Tier1').astype(float)
    + 0.15 * (employed == 0).astype(float)
    + 0.10 * (emi_income > 0.45).astype(float)
    + 0.12 * (missed_pmts_past > 0).astype(float)
    - 0.05 * auto_debit.astype(float)
    - 0.04 * co_borrower.astype(float)
    + 0.06 * (months_since_disbursement <= 6).astype(float)
    + np.random.normal(0, 0.03, N)
).clip(0.01, 0.90)

default_30d = (
    np.random.uniform(0, 1, N) < base_risk * 0.6
).astype(int)

default_60d = np.maximum(
    default_30d,
    (
        np.random.uniform(0, 1, N) < base_risk * 0.8
    ).astype(int)
)

default_90d = np.maximum(
    default_60d,
    (
        np.random.uniform(0, 1, N) < base_risk
    ).astype(int)
)

df = pd.DataFrame({
    'loan_id': [f"SL{i:08d}" for i in range(N)],
    'age': age,
    'gender': gender,
    'course_type': course_type,
    'institute_tier': institute_tier,
    'gpa': gpa,
    'gpa_trend': gpa_trend,
    'loan_amt': loan_amt.astype(int),
    'interest_rate': interest_rate,
    'tenure_mo': tenure_mo,
    'moratorium_mo': moratorium_mo,
    'emi': emi.astype(int),
    'employed': employed,
    'salary_mo': salary_mo.astype(int),
    'emi_income': emi_income,
    'months_since_disb': months_since_disbursement,
    'missed_pmts_past': missed_pmts_past,
    'payment_delay_avg': payment_delay_avg,
    'auto_debit': auto_debit,
    'co_borrower': co_borrower,
    'family_income_mo': family_income_mo.astype(int),
    'default_30d': default_30d,
    'default_60d': default_60d,
    'default_90d': default_90d
})

df.to_csv('../data/student_loans.csv', index=False)

print(f"Shape: {df.shape}")
print(f"Default 30d rate: {default_30d.mean()*100:.1f}%")
print(f"Default 60d rate: {default_60d.mean()*100:.1f}%")
print(f"Default 90d rate: {default_90d.mean()*100:.1f}%")
print(f"Employment rate: {employed.mean()*100:.1f}%")

print("Saved: data/student_loans.csv")

Generating Student Loan dataset...
Shape: (60000, 24)
Default 30d rate: 9.5%
Default 60d rate: 20.3%
Default 90d rate: 31.3%
Employment rate: 72.1%
Saved: data/student_loans.csv
